<a href="https://colab.research.google.com/github/whit7990/usports-fb-xml-check/blob/main/XML_Play_Validator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title 0. Upload Team Rosters
from google.colab import files
import re

def parse_pasted_roster(raw_text):
    """
    Universally parses athletic rosters, handling both vertical stacked formats
    (Number -> Name -> Position on separate lines) and horizontal single-line formats.
    """
    team_map = {}
    if not raw_text:
        return team_map

    lines = raw_text.strip().split('\n')

    known_positions = {
        'QB', 'RB', 'FB', 'WR', 'SB', 'DB', 'CB', 'LB', 'DL', 'OL',
        'K', 'P', 'LS', 'TE', 'S', 'K/P', 'K/WR', 'WR/RET', 'REC'
    }

    current_num = ""
    current_name = ""

    for line in lines:
        line = line.strip().replace('\t', ' ')
        if not line or line.startswith('#') or 'Football' in line:
            continue

        tokens = line.split()
        if not tokens:
            continue

        found_pos = ""
        pos_idx = -1
        for idx, token in enumerate(tokens):
            clean_token = token.upper().strip().rstrip(',')
            if clean_token in known_positions:
                found_pos = clean_token
                pos_idx = idx
                break

        if found_pos and pos_idx > 0:
            has_num = bool(re.match(r'^\d{1,2}$', tokens[0]))
            num_val = tokens[0] if has_num else ""
            start_idx = 1 if has_num else 0
            name_tokens = tokens[start_idx:pos_idx]
            name_tokens = [t for t in name_tokens if not re.match(r'^\d+[\'"]?\d*$', t) and t.upper() not in ['POS', 'ELIG', 'HT', 'WT', 'YR']]

            if name_tokens:
                player_name = " ".join(name_tokens).strip()
                player_name = re.sub(r'[^a-zA-Z\s\-\.\']', '', player_name).strip()

                if len(player_name) > 2 and player_name.lower() not in ['name', 'pos', 'elig', 'ht', 'major', 'hometown', 'first', 'last']:
                    team_map[player_name.lower()] = {
                        'position': found_pos,
                        'number': num_val
                    }
                    current_num = ""
                    current_name = ""
                    continue

        if re.match(r'^\d{1,2}$', line) and len(tokens) == 1:
            current_num = line
            current_name = ""
            continue

        if current_num and not current_name:
            if line.lower() not in ['no.', 'name', 'pos.', 'elig.', 'ht.', 'hometown / high school or prior team', 'major']:
                current_name = line
            continue

        if current_num and current_name:
            for token in tokens:
                clean_token = token.upper().strip().rstrip(',')
                if clean_token in known_positions:
                    clean_name = re.sub(r'[^a-zA-Z\s\-\.\']', '', current_name).strip()
                    if len(clean_name) > 2:
                        team_map[clean_name.lower()] = {
                            'position': clean_token,
                            'number': current_num
                        }
                    current_num = ""
                    current_name = ""
                    break

    return team_map

print("Upload the Visitor roster (.txt file with pasted roster text):")
visitor_upload = files.upload()
visitor_text = list(visitor_upload.values())[0].decode('utf-8')

print("Upload the Home roster (.txt file with pasted roster text):")
home_upload = files.upload()
home_text = list(home_upload.values())[0].decode('utf-8')

web_rosters = {
    'Visitor': parse_pasted_roster(visitor_text),
    'Home': parse_pasted_roster(home_text)
}

print(f"Successfully loaded Visitor roster: {len(web_rosters['Visitor'])} players found.")
print(f"Successfully loaded Home roster: {len(web_rosters['Home'])} players found.")

Upload the Visitor roster (.txt file with pasted roster text):


Saving Otawa.txt to Otawa (1).txt
Upload the Home roster (.txt file with pasted roster text):


Saving Laurier.txt to Laurier (1).txt
Successfully loaded Visitor roster: 62 players found.
Successfully loaded Home roster: 114 players found.


In [3]:
#@title 1. Upload XML File
from google.colab import files

print("Please upload your PrestoSports Stat Crew XML file:")
uploaded = files.upload()
xml_filename = list(uploaded.keys())[0]
print(f"Successfully loaded: {xml_filename}\n")

Please upload your PrestoSports Stat Crew XML file:


Saving boxscore_20260912_0636.xml to boxscore_20260912_0636 (1).xml
Successfully loaded: boxscore_20260912_0636 (1).xml



In [6]:
#@title 2. Parse XML and Build Roster Dictionary
import xml.etree.ElementTree as ET

# Parse XML root
tree = ET.parse(xml_filename)
root = tree.getroot()

roster = {}
team_counts = {'Visitor': 0, 'Home': 0}

for team_tag in ['visiting-team', 'home-team', 'team']:
    for team in root.findall(f".//{team_tag}"):
        # Determine Visitor/Home from the tag itself, or from the 'vh'
        # attribute for the generic <team> tag used by this file format.
        if team_tag == 'visiting-team':
            team_label = 'Visitor'
        elif team_tag == 'home-team':
            team_label = 'Home'
        else:
            vh_attr = team.get('vh', '').upper()
            team_label = 'Visitor' if vh_attr == 'V' else 'Home' if vh_attr == 'H' else team.get('name', 'Unknown')

        team_id = team.get('id') or team.get('code') or team_label
        matched_web_roster = web_rosters.get(team_label, {})

        for player in team.findall('.//player'):
            p_name = player.get('name', '').strip()
            p_uni = player.get('uni', '')

            # Use the XML's own globally-unique playerId when available;
            # otherwise fall back to a TEAM-SCOPED key (team_id + uni) so
            # two different teams' players sharing a jersey number never
            # overwrite each other -- which is what caused the undercount.
            pid = player.get('playerId') or f"{team_id}_{p_uni}"

            # Default position from XML if available
            pos = player.get('pos', '').upper()

            # Match by name first (case-insensitive)
            if p_name.lower() in matched_web_roster:
                pos = matched_web_roster[p_name.lower()]['position']
            elif p_uni:
                # Fallback check by number if name fails
                for w_name, w_info in matched_web_roster.items():
                    if w_info['number'] == p_uni:
                        pos = w_info['position']
                        break

            if pid:
                roster[pid] = {
                    'uni': p_uni,
                    'pos': pos,
                    'name': p_name,
                    'team': team_label
                }
                team_counts[team_label] = team_counts.get(team_label, 0) + 1

print(f"Roster loaded successfully. Total players indexed: {len(roster)}")
print(f"  Visitor: {team_counts.get('Visitor', 0)} players indexed")
print(f"  Home:    {team_counts.get('Home', 0)} players indexed")

Roster loaded successfully. Total players indexed: 58
  Visitor: 28 players indexed
  Home:    30 players indexed


In [7]:
#@title 3. Audit Plays and Run QC Rules
import re

# Standard Canadian Penalty Yardage & Code Reference Map
# Distances: int = single fixed standard. None = no single fixed
# distance (situational/variable per the official code sheet) -- these
# are still recognized codes, but yardage mismatches aren't auto-checked.
CANADIAN_PENALTIES = {
    'O2':  (0,    '3 OC Penalties - Disqualification'),
    'U3':  (0,    '3 UR Penalties - Disqualification'),
    'BSB': (15,   'Blindside Block'),
    'BST': (25,   'Blindside Block with Targeting'),
    'BW':  (10,   'Blocking Below the Waist'),
    'BR':  (15,   'Blocking from the Rear (Clipping)'),
    'CK':  (10,   'Contacting the Kicker'),
    'CB':  (15,   'Crackback Block'),
    'DG':  (10,   'Delay of Game'),
    'DK':  (15,   'Delayed Knee Block'),
    'DR':  (None, 'Discriminatory Remarks'),
    'DD':  (None, 'Discriminatory Remarks - Disqualification'),
    'FM':  (15,   'Face Mask'),
    'HTF': (10,   'Hands to the Face'),
    'HO':  (10,   'Holding'),
    'HCT': (15,   'Horse Collar Tackle'),
    'BL':  (10,   'Illegal Block'),
    'CR':  (10,   'Illegal Contact on an Eligible Receiver'),
    'EQ':  (5,    'Illegal Equipment'),
    'FP':  (10,   'Illegal Forward Pass'),
    'HC':  (15,   'Illegal Helmet Contact (Head Tackle)'),
    'LB':  (0,    'Illegal Interference on a Loose Ball'),
    'IK':  (10,   'Illegal Interference on the Kicker'),
    'KO':  (0,    'Illegal Kickoff'),
    'PA':  (10,   'Illegal Participation'),
    'IP':  (5,    'Illegal Procedure'),
    'ILR': (10,   'Illegal Receiver'),
    'IS':  (10,   'Illegal Substitution'),
    'UH':  (10,   'Illegal Use of Hands'),
    'IRD': (10,   'Ineligible Receiver Downfield'),
    'IG':  (None, 'Intentional Grounding (loss of down)'),
    'GG':  (None, 'Interference by Unauthorized Persons'),
    'MG':  (5,    'No Mouthguard'),
    'MW':  (0,    'No Mouthguard Warning'),
    'NY':  (None, 'No Yards (5 or 15 depending on situation)'),
    'OC':  (10,   'Objectionable Conduct'),
    'XOC': (0,    'Objectionable Conduct - Disqualification'),
    'OS':  (5,    'Offside'),
    'PI':  (None, 'Pass Interference (0-15, spot foul)'),
    'PO':  (15,   'Piling On'),
    'RD':  (25,   'Rough Play - Disqualification'),
    'RH':  (15,   'Roughing the Holder'),
    'RK':  (15,   'Roughing the Kicker'),
    'RP':  (15,   'Roughing the Passer'),
    'RS':  (15,   'Roughing the Snapper'),
    'SP':  (15,   'Spearing (Head-Leading Tackle/Block)'),
    'TB':  (5,    'Illegal Procedure - Tandem Buck'),
    'TGT': (25,   'Targeting'),
    'TD':  (25,   'Targeting - Disqualification'),
    'TC':  (None, 'Time Count Violation (5, 10, or loss of down)'),
    'TH':  (5,    'Too Many Men in the Huddle'),
    '12':  (10,   'Too Many Men on the Field'),
    'TR':  (10,   'Tripping'),
    'UNR': (15,   'Unnecessary Roughness'),
}

errors = []
warnings = []
pass_rec_results = []
total_plays = 0

def check_drivestart_clock_advanced(qtr_children, play_element, play_clock):
    """
    Looks immediately after `play_element` in the quarter's full child list
    (which includes <drivesum>/<drivestart>/<score>, unlike the play-only
    list used elsewhere) for a <drivestart> tag that appears before any
    further <play> tag. If found, and its clock differs from the play's
    own clock, this is authoritative confirmation that the clock genuinely
    advanced at the start of the next drive -- even if adjacent play-tag
    records still show the old time. Returns False if a new <play> is
    encountered first, since that means we have no direct confirmation.
    """
    try:
        start_idx = qtr_children.index(play_element)
    except ValueError:
        return False
    for sibling in qtr_children[start_idx + 1:]:
        if sibling.tag == 'play':
            return False
        if sibling.tag == 'drivestart':
            ds_clock = sibling.get('clock', '')
            return bool(ds_clock) and ds_clock != play_clock
    return False

def extract_play_number(play_tag):
    """
    Extracts the 3rd number strictly from the playid attribute value.
    Example: playid='FBK-2026-Q1-P12' -> extracts '12' as the 3rd number.
    """
    play_id_val = play_tag.get('playid', '')
    if play_id_val:
        numbers = re.findall(r'\d+', play_id_val)
        if len(numbers) >= 3:
            return numbers[2]
    return "N/A"

def extract_starting_yardline(play_tag):
    """
    Extracts the starting yard line directly from the play's 'context'
    attribute (e.g. context='V,1,10,V45' -> 'V45'), which already encodes
    the correct V/H side from the source system. This avoids inferring the
    side from free-text play descriptions, which broke on any game where
    the visiting team wasn't literally named 'GUELPH' or 'VISITOR'.
    Falls back to 'H00' if context is missing or malformed.
    """
    context = play_tag.get('context', '')
    if context:
        last_field = context.split(',')[-1].strip()
        match = re.match(r'^([VH])(\d{1,2})$', last_field)
        if match:
            side, yard = match.groups()
            return f"{side}{yard.zfill(2)}"
    return "H00"

def get_roster_player_info(player_name):
    """
    Checks player name against web_rosters parsed in Cell 0B.
    Returns (position, team) or (None, None).
    """
    if not player_name or 'web_rosters' not in globals():
        return None, None
    p_lower = player_name.strip().lower()
    for team, roster in web_rosters.items():
        if p_lower in roster:
            return roster[p_lower]['position'], team
    return None, None

def player_has_position(pos_field, target_positions):
    """
    A player's roster position can list multiple valid positions separated
    by '/' (e.g. 'K/WR', 'QB/WR', 'K/P'). This checks whether ANY of the
    listed positions match the target set, rather than treating the whole
    field as one atomic code -- so a K/WR performing a kick, or a QB/WR
    throwing a pass, isn't flagged as a mismatch.
    """
    if not pos_field:
        return False
    components = [p.strip() for p in pos_field.upper().split('/')]
    return any(comp in target_positions for comp in components)

# --- POSITION ANOMALY AUDIT HELPER ---
def check_player_position_anomalies(play_element, warnings_list, web_rosters, play_num):
    """
    Audits Stat Crew XML play elements against web_rosters for:
    1. Passing stats by non-QBs.
    2. Offensive stats by defensive players (DB, LB, DL, S).
    3. Kicking stats by non-kickers/punters (K, P, K/P).
    Appends structured dictionaries compatible with Cell 4 markdown output.
    """
    defensive_positions = {'DB', 'LB', 'DL', 'S', 'CB', 'DE'}
    kicking_positions = {'K', 'P'}

    play_type = play_element.get('type', '').upper()
    text_content = play_element.get('text', '')

    # 1. Check Passing Actions (<p_pa> tag or type 'P')
    pa_elem = play_element.find('p_pa')
    if play_type == 'P' or pa_elem is not None:
        qb_name = pa_elem.get('qb') if pa_elem is not None else None
        if not qb_name:
            text_match = re.search(r'^([A-Za-z\s\.\-\']+)\s+pass', text_content)
            if text_match:
                qb_name = text_match.group(1).strip()

        if qb_name:
            pos, team = get_roster_player_info(qb_name)
            if pos and not player_has_position(pos, {'QB'}):
                warnings_list.append({
                    'play_num': play_num,
                    'details': text_content,
                    'issue': f"Pass Mismatch: {qb_name} ({team}, {pos}) completed/threw a pass but is not listed as a QB."
                })

    # 2. Check Offensive Stats by Defensive Players
    for team_label, roster in web_rosters.items():
        for p_name, info in roster.items():
            if p_name in text_content.lower() and text_content.lower().startswith(p_name):
                pos = info['position']
                if player_has_position(pos, defensive_positions):
                    if 'rush' in text_content.lower() or 'pass complete to' in text_content.lower() or 'reception' in text_content.lower():
                        warnings_list.append({
                            'play_num': play_num,
                            'details': text_content,
                            'issue': f"Defense-Offense Mismatch: {p_name.title()} ({team_label}, {pos}) recorded an offensive action."
                        })

    # 3. Check Kicking Stats by Non-Kickers
    # Only check the player actually credited as the kicker via the
    # sub-element's own 'name' attribute (p_ko/p_fg/p_pu) -- not every
    # name that happens to appear in the play text, which would also
    # catch the returner and any tackler listed in parentheses.
    kick_elem = play_element.find('p_ko')
    if kick_elem is None:
        kick_elem = play_element.find('p_fg')
    if kick_elem is None:
        kick_elem = play_element.find('p_pu')
    if kick_elem is not None:
        kicker_name = kick_elem.get('name', '')
        if kicker_name:
            pos, team = get_roster_player_info(kicker_name)
            if pos and not player_has_position(pos, kicking_positions):
                warnings_list.append({
                    'play_num': play_num,
                    'details': text_content,
                    'issue': f"Kicking Mismatch: {kicker_name} ({team}, {pos}) executed a kicking action but is not a K/P."
                })
# Audit Plays loop (iterating through each quarter)
for qtr in root.findall('.//qtr'):
    qtr_num = qtr.get('number', '1')
    plays_in_qtr = qtr.findall('play')
    qtr_children = list(qtr)  # full sibling list, including drivesum/drivestart/score
    num_plays = len(plays_in_qtr)

    for i in range(num_plays):
        play = plays_in_qtr[i]
        total_plays += 1
        raw_play_num = extract_play_number(play)
        clock = play.get('clock', '')
        text = play.get('text', '')
        text_lower = text.lower()
        score_flag = play.get('score', 'N')

        start_yl = extract_starting_yardline(play)
        play_num = f"{raw_play_num} ({start_yl})"

        play_desc = f"Q{qtr_num} {clock if clock else 'No Clock'} - {text}"
        is_noplay = 'no play' in text_lower or 'noplay' in text_lower

# --- CHECK 1: Clock & Possession Change Checks ---
        if not is_noplay:
            is_scoring = (score_flag == 'Y' or play.find('scores') is not None)
            is_touchdown = 'touchdown' in text_lower
            is_pat_convert = any(term in text_lower for term in ['kick attempt good', 'convert good', 'point attribute'])

            # A turnover/kick is a possession change, but a pure touchdown play is a score (possession changes on kickoff)
            is_actual_possession_change = any(term in text_lower for term in ['intercept', 'fumble lost', 'turnover on downs', 'punt', 'kickoff return', 'field goal attempt', 'missed'])

            should_check_clock = (is_scoring and not is_touchdown) or (is_actual_possession_change and not is_pat_convert)

            if should_check_clock:
                if not clock:
                    errors.append({
                        'play_num': play_num,
                        'details': play_desc,
                        'issue': "Clock time is missing on a scoring play or change of possession/kick sequence."
                    })
                else:
                    next_clock = plays_in_qtr[i + 1].get('clock', '') if i + 1 < num_plays else ''
                    prev_clock = plays_in_qtr[i - 1].get('clock', '') if i - 1 >= 0 else ''

                    if clock == prev_clock and (not next_clock or clock == next_clock):
                        if check_drivestart_clock_advanced(qtr_children, play, clock):
                            warnings.append({
                                'play_num': play_num,
                                'details': play_desc,
                                'issue': "Clock time appears static across adjacent play records, but the next drive-start record shows a different clock time, confirming the game clock did in fact advance."
                            })
                        else:
                            errors.append({
                                'play_num': play_num,
                                'details': play_desc,
                                'issue': "Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence."
                            })

            if clock:
                try:
                    parts = clock.split(':')
                    minutes = int(parts[0])
                    if minutes > 15 or minutes < 0:
                        errors.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Clock value {clock} exceeds 15:00 regulation limit or is negative."
                        })
                except ValueError:
                    pass

        # --- CHECK 2: Penalty Code & Yardage Validation (with Inside-15 Yardline Note) ---
        numeric_yl = None
        yl_side_match = re.match(r'^[VH](\d{1,2})$', start_yl)
        if yl_side_match:
            numeric_yl = int(yl_side_match.group(1))

        is_inside_fifteen = (numeric_yl is not None and 1 <= numeric_yl <= 15)

        for pn in play.findall('.//p_pn'):
            accept_status = pn.get('intent', pn.get('accept', '')).lower()
            if accept_status in ['no', 'false', 'declined'] or 'declined' in text_lower:
                continue

            code = pn.get('code', '').upper()
            try:
                yards_assessed = int(pn.get('yards', 0))
            except (ValueError, TypeError):
                yards_assessed = 0

            if code in ['PF', 'UC']:
                errors.append({
                    'play_num': play_num,
                    'details': play_desc,
                    'issue': f"American penalty code/term '{code}' used. Must use Canadian terminology."
                })
                continue

            # Look up a friendly name where we recognize the code; otherwise
            # flag it generically so unrecognized codes still get reviewed.
            penalty_name = CANADIAN_PENALTIES[code][1] if (code in CANADIAN_PENALTIES and CANADIAN_PENALTIES[code]) else None

            # --- NEW: flag any accepted penalty assessed exactly 0 yards,
            # unless 0 yards is the documented standard for that code
            # (e.g. disqualification-only penalties like O2/U3/XOC, or
            # KO/LB/MW which are officially 0 yards per the code sheet).
            documented_zero = code in CANADIAN_PENALTIES and CANADIAN_PENALTIES[code] and CANADIAN_PENALTIES[code][0] == 0
            if yards_assessed == 0 and not documented_zero:
                display_name = f"{code} ({penalty_name})" if penalty_name else f"{code} (unrecognized code)"
                if is_inside_fifteen:
                    warnings.append({
                        'play_num': play_num,
                        'details': play_desc,
                        'issue': f"Penalty {display_name} was assessed 0 yards. Note: Play started inside the 15-yard line ({start_yl}). Verify this is not a missing/omitted yardage entry."
                    })
                else:
                    warnings.append({
                        'play_num': play_num,
                        'details': play_desc,
                        'issue': f"Penalty {display_name} was assessed 0 yards. Verify this is not a missing/omitted yardage entry."
                    })
            elif code in CANADIAN_PENALTIES and CANADIAN_PENALTIES[code]:
                std_yards, _ = CANADIAN_PENALTIES[code]
                if isinstance(std_yards, int) and std_yards != 0 and yards_assessed != std_yards:
                    if is_inside_fifteen:
                        warnings.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Penalty {code} ({penalty_name}) assessed {yards_assessed} yards (standard is {std_yards}). Note: Play started inside the 15-yard line ({start_yl})."
                        })
                    else:
                        warnings.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': f"Penalty {code} ({penalty_name}) assessed {yards_assessed} yards; standard Canadian distance is {std_yards} yards."
                        })

        # --- CHECK 3: Check positions
        check_player_position_anomalies(play, warnings, web_rosters, play_num)

        # --- CHECK 5: Large Yardage Loss Detection (>= 20 yards) ---
        loss_patterns = [
            r'loss of (\d{2,})',
            r'sacked for (-?\d{2,})',
            r'for loss of (\d{2,})',
            r'\b(-\d{2,})\b'
        ]

        large_loss_found = False
        loss_val_detected = 0
        for pattern in loss_patterns:
            match = re.search(pattern, text_lower)
            if match:
                try:
                    val = abs(int(match.group(1)))
                    if val >= 20:
                        large_loss_found = True
                        loss_val_detected = val
                        break
                except ValueError:
                    pass

        for subelem in play.iter():
            for attr in ['loss', 'sackyds', 'yds', 'gain']:
                val_str = subelem.get(attr, '')
                if val_str:
                    try:
                        v = int(val_str)
                        if v <= -20 or (attr in ['loss', 'sackyds'] and v >= 20):
                            large_loss_found = True
                            loss_val_detected = abs(v)
                    except ValueError:
                        pass

        if large_loss_found:
            warnings.append({
                'play_num': play_num,
                'details': play_desc,
                'issue': f"Large yardage loss detected (approx. {loss_val_detected} yards). Please verify play metrics."
            })

# --- CHECK 6: Unauthorized TEAM / TM Stat Attribution & Team Rush Losses ---
        is_team_safety = 'safety' in text_lower and ('team' in text_lower or 'tm' in text_lower)
        play_flagged_for_team_stat = False

        if not is_team_safety:
            # 1. Check text-based team attributions (e.g., tackles by (TEAM; TEAM))
            if re.search(r'\((TEAM|TM);\s*(TEAM|TEAM)\)', text, re.IGNORECASE):
                errors.append({
                    'play_num': play_num,
                    'details': play_desc,
                    'issue': "TEAM or TM should not be given stats: generic team entity '(TEAM; TEAM)' found in play text attribution."
                })
                play_flagged_for_team_stat = True

            # 2. Check all sub-elements for TEAM/TM stats (only if not already flagged)
            if not play_flagged_for_team_stat:
                for subelem in play.iter():
                    s_name = subelem.get('name', '').upper()
                    s_uni = subelem.get('uni', '').upper()

                    if s_name in ['TEAM', 'TM'] or s_uni in ['TEAM', 'TM']:
                        has_ff = (subelem.get('ff', '').upper() == 'Y')
                        has_stats = any(subelem.find(tag) is not None for tag in ['rush', 'pass', 'rcv', 'tackle', 'fum', 'forced'])
                        has_inline_attr = any(subelem.get(attr) not in [None, '0', '', 'N'] for attr in ['ff', 'forced', 'fumble_forced', 'tackles'])

                        if has_ff or has_stats or has_inline_attr or subelem.tag in ['p_tk', 'p_fum']:
                            errors.append({
                                'play_num': play_num,
                                'details': play_desc,
                                'issue': f"TEAM or TM should not be given stats: statistics incorrectly credited to generic entity '{s_name or s_uni}' via tag <{subelem.tag}>."
                            })
                            play_flagged_for_team_stat = True
                            break

            # 3. Check standalone fumble elements for TEAM/TM
            if not play_flagged_for_team_stat:
                for fum_elem in play.findall('.//fum'):
                    fum_forced_by = fum_elem.get('forced_by', '') or fum_elem.get('ff_by', '')
                    if fum_forced_by.upper() in ['TEAM', 'TM']:
                        errors.append({
                            'play_num': play_num,
                            'details': play_desc,
                            'issue': "TEAM or TM should not be given stats: forced fumble statistic incorrectly attributed to TEAM/TM instead of an individual player."
                        })
                        play_flagged_for_team_stat = True
                        break

            # 4. Check rush for loss credited to team
            if not play_flagged_for_team_stat and 'rush' in text_lower and ('loss' in text_lower or 'sacked' in text_lower or '(-' in text_lower):
                for sub in play.findall('.//rush'):
                    try:
                        yds = int(sub.get('yds', 0))
                        loss_val = int(sub.get('loss', 0))
                        if yds < 0 or loss_val > 0:
                            if 'team' in str(sub.attrib).lower() or 'tm' in text_lower:
                                errors.append({
                                    'play_num': play_num,
                                    'details': play_desc,
                                    'issue': "TEAM or TM should not be given stats: rush for a loss incorrectly credited as a team-level stat instead of an individual player."
                                })
                                break
                    except ValueError:
                        pass

# --- PASSING VS. RECEIVING YARDS VALIDATION (Per Team) ---
team_nodes = root.findall('.//team')
team_stats = {}

for i, team in enumerate(team_nodes):
    vh_attr = team.get('vh', '').upper()
    if vh_attr == 'H' or (i == 1 and not vh_attr):
        team_label = "Home"
    elif vh_attr == 'V' or (i == 0 and not vh_attr):
        team_label = "Visitor"
    else:
        team_label = team.get('name', f"Team {i+1}")

    team_stats[id(team)] = {
        'label': team_label,
        'team_node': team,
        'pass_yds': 0,
        'rcv_yds': 0
    }

player_to_team_id = {}
for t_id, data in team_stats.items():
    t_node = data['team_node']
    for player in t_node.iter('player'):
        player_to_team_id[id(player)] = t_id

for player in root.iter('player'):
    t_id = player_to_team_id.get(id(player))
    if t_id and t_id in team_stats:
        pass_tag = player.find('pass')
        if pass_tag is not None:
            try:
                team_stats[t_id]['pass_yds'] += int(pass_tag.get('yds', 0))
            except ValueError:
                pass

        for rcv_tag_name in ['rcv', 'receiving']:
            rcv_tag = player.find(rcv_tag_name)
            if rcv_tag is not None:
                try:
                    team_stats[t_id]['rcv_yds'] += int(rcv_tag.get('yds', 0))
                except ValueError:
                    pass

for stats in team_stats.values():
    t_label = stats['label']
    p_yds = stats['pass_yds']
    r_yds = stats['rcv_yds']

    if p_yds == r_yds:
        pass_rec_results.append(f"* **{t_label}**: [PASS] Passing Yards ({p_yds}) match Receiving Yards ({r_yds}).")
    else:
        pass_rec_results.append(f"* **{t_label}**: [ERROR] Passing Yards ({p_yds}) do not match Receiving Yards ({r_yds}).")

# --- PLAYER STAT ENTRY LOGIC VALIDATION ---
for player in root.iter('player'):
    name = player.get('name', 'Unknown')
    uni = player.get('uni', 'N/A')

    if name.upper() in ['TEAM', 'TM'] or uni.upper() in ['TEAM', 'TM']:
        continue

    pass_tag = player.find('pass')
    if pass_tag is not None:
        try:
            att = int(pass_tag.get('att', 0))
            comp = int(pass_tag.get('comp', 0))
            if comp > att:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Passing Stats Record -> Attempts: {att}, Completions: {comp}",
                    'issue': f"Completions ({comp}) exceed passing attempts ({att})."
                })
            if att < 0 or comp < 0:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Passing Stats Record -> Attempts: {att}, Completions: {comp}",
                    'issue': 'Negative passing attempts or completions found.'
                })
        except ValueError:
            pass

    rush_tag = player.find('rush')
    if rush_tag is not None:
        try:
            att = int(rush_tag.get('att', 0))
            if att < 0:
                errors.append({
                    'play_num': f"Player #{uni} ({name})",
                    'details': f"Rushing Stats Record -> Attempts: {att}",
                    'issue': f"Negative rushing attempts ({att}) found."
                })
        except ValueError:
            pass

print(f"Audit complete. Reviewed {total_plays} plays and all player stat records.")

Audit complete. Reviewed 168 plays and all player stat records.


In [8]:
#@title 4. Output Structured Report
from IPython.display import display, Markdown

def parse_top_to_seconds(top_str):
    """Converts a TOP string (e.g., '28:15') into total seconds."""
    if not top_str or ':' not in top_str:
        return 0
    try:
        parts = top_str.split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except (ValueError, IndexError):
        return 0

def format_seconds_to_mmss(total_seconds):
    """Converts total seconds back into MM:SS format."""
    m = total_seconds // 60
    s = total_seconds % 60
    return f"{m:02d}:{s:02d}"

# --- TIME OF POSSESSION (TOP) COMBINED GAME TOTAL CHECK ---
quarters_found = root.findall('.//qtr')
max_qtr = 0
for qtr in quarters_found:
    try:
        q_num = int(qtr.get('number', '0'))
        if q_num > max_qtr:
            max_qtr = q_num
    except ValueError:
        pass

expected_total_seconds = 1800 if max_qtr <= 2 else 3600
expected_label = "30 minutes (half)" if max_qtr <= 2 else "60 minutes (full game)"

combined_top_seconds = 0
team_top_records = []

misc_nodes = root.findall('.//misc')
for i, misc in enumerate(misc_nodes):
    vh_attr = misc.get('vh', '').upper()
    if vh_attr == 'H' or (i == 1 and not vh_attr):
        team_label = "Home"
    elif vh_attr == 'V' or (i == 0 and not vh_attr):
        team_label = "Visitor"
    else:
        team_label = misc.get('team', f"Team {i+1}")

    top_val = misc.get('top', '')
    if top_val:
        secs = parse_top_to_seconds(top_val)
        combined_top_seconds += secs
        team_top_records.append(f"{team_label}: {top_val}")

diff_seconds = abs(combined_top_seconds - expected_total_seconds)
combined_str = format_seconds_to_mmss(combined_top_seconds)

# Generate Structured Markdown Output
markdown_output = []
markdown_output.append("=" * 60)
markdown_output.append("### CANADIAN FOOTBALL XML QUALITY CONTROL REPORT")
markdown_output.append("=" * 60)

# TOP Section
markdown_output.append("\n## TIME OF POSSESSION")
if team_top_records:
    markdown_output.append(f"* **Team Breakdown**: " + " | ".join(team_top_records))
    markdown_output.append(f"* **Combined Game Total**: {combined_str} (Expected: {expected_label})")
    if diff_seconds <= 5:
        markdown_output.append("* **Status**: [TOP PASS] Combined Time of Possession matches expected duration.")
    else:
        markdown_output.append(f"* **Status**: [ERROR] Combined Time of Possession ({combined_str}) does not equal expected {expected_label}.")
else:
    markdown_output.append("* **Status**: [TOP NOTICE] No team 'misc' statistics nodes with 'top' attributes found in XML.")

# Passing vs Receiving Section
markdown_output.append("\n## PASSING VS. RECEIVING YARDS")
if pass_rec_results:
    for res in pass_rec_results:
        markdown_output.append(res)
else:
    markdown_output.append("* **Status**: [NOTICE] No team passing/receiving statistics nodes found in XML.")

markdown_output.append(f"\n## ERROR ({len(errors)})")
if not errors:
    markdown_output.append("* No errors found.")
else:
    for err in errors:
        markdown_output.append(f"* **Play #**: {err['play_num']}")
        markdown_output.append(f"  * **Play Details**: {err['details']}")
        markdown_output.append(f"  * **Issue Identified**: {err['issue']}")

markdown_output.append(f"\n## WARNING ({len(warnings)})")
if not warnings:
    markdown_output.append("* No warnings found.")
else:
    for warn in warnings:
        markdown_output.append(f"* **Play #**: {warn['play_num']}")
        markdown_output.append(f"  * **Play Details**: {warn['details']}")
        markdown_output.append(f"  * **Issue Identified**: {warn['issue']}")

markdown_output.append("\n" + "-" * 60)
markdown_output.append(f"**Total plays reviewed**: {total_plays}")

# Join and display using IPython Markdown display tool
full_markdown_str = "\n".join(markdown_output)
display(Markdown(full_markdown_str))

============================================================
### CANADIAN FOOTBALL XML QUALITY CONTROL REPORT
============================================================

## TIME OF POSSESSION
* **Team Breakdown**: Visitor: 36:23 | Home: 23:37
* **Combined Game Total**: 60:00 (Expected: 60 minutes (full game))
* **Status**: [TOP PASS] Combined Time of Possession matches expected duration.

## PASSING VS. RECEIVING YARDS
* **Visitor**: [PASS] Passing Yards (245) match Receiving Yards (245).
* **Home**: [PASS] Passing Yards (367) match Receiving Yards (367).

## ERROR (2)
* **Play #**: 138 (V37)
  * **Play Details**: Q4 10:07 - TEAM field goal attempt from 44 MISSED - short, spot at OTT37, clock 10:07.
  * **Issue Identified**: Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence.
* **Play #**: 161 (V45)
  * **Play Details**: Q4 01:10 - Marcus Reypa field goal attempt from 62 MISSED, kick to OTT-10, clock 01:10, Denny Ferdinand return 29 yards to the OTT19 (Zidain Allen), clock 01:10.
  * **Issue Identified**: Clock time remains static (unchanged across adjacent plays) on a major possession change or kick sequence.

## WARNING (7)
* **Play #**: 29 (H47)
  * **Play Details**: Tayshaun Jackson pass complete to Ryan Speight for 26 yards to the OTT37 (Maxim Piché; James Lannon), clock 05:09.
  * **Issue Identified**: Pass Mismatch: Tayshaun Jackson (Home, RB) completed/threw a pass but is not listed as a QB.
* **Play #**: 58 (H54)
  * **Play Details**: Q2 06:44 - Zachary Copeland punt 45 yards to the WLU09, Logan Moore return 21 yards to the WLU30, out-of-bounds, PENALTY WLU holding 19 yards to the WLU11, clock 06:44.
  * **Issue Identified**: Penalty HO (Holding) assessed 19 yards; standard Canadian distance is 10 yards.
* **Play #**: 115 (H02)
  * **Play Details**: Nassim Regragui rush for loss of 1 yard to the WLU03 (Oscar Wise), clock 02:47.
  * **Issue Identified**: Defense-Offense Mismatch: Nassim Regragui (Visitor, LB) recorded an offensive action.
* **Play #**: 117 (H03)
  * **Play Details**: Q3 00:37 - Josh Janssen pass incomplete to Tristan Gilbert Thibault (Tucker Pinet), PENALTY WLU cr 2 yards to the WLU01, NO PLAY, clock 00:37.
  * **Issue Identified**: Penalty CR (Illegal Contact on an Eligible Receiver) assessed 2 yards (standard is 10). Note: Play started inside the 15-yard line (H03).
* **Play #**: 133 (V35)
  * **Play Details**: Q4 10:07 - Quinten Springer rush for loss of 32 yards to the WLU43 (Marc Rondeau), clock 10:07.
  * **Issue Identified**: Large yardage loss detected (approx. 32 yards). Please verify play metrics.
* **Play #**: 136 (V37)
  * **Play Details**: Thomas Frizzarin pass incomplete, clock 10:07.
  * **Issue Identified**: Pass Mismatch: Thomas Frizzarin (Visitor, TE) completed/threw a pass but is not listed as a QB.
* **Play #**: 137 (V37)
  * **Play Details**: Thomas Frizzarin pass incomplete, clock 10:07.
  * **Issue Identified**: Pass Mismatch: Thomas Frizzarin (Visitor, TE) completed/threw a pass but is not listed as a QB.

------------------------------------------------------------
**Total plays reviewed**: 168